# D2.1 · Agent-assisted reconstruction

**Function D — Security Operations → The Incident Responder**  ·  *AI for Security*

---

**Risk.** Reaching for the agent once you're already behind.

**Control.** Pre-load logs, telemetry, segmentation model and playbooks.

**This lab.** Pre-load the agent so it reasons as a partner, not a tool you reach for late.

| | |
|---|---|
| Open-source tooling | Velociraptor, OpenSearch |
| Open-weight models | GLM-4.6 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("D2.1"))

Agent-assisted reconstruction is genuinely faster — provided the evidence is there. The failure mode is a confident narrative built on logs that were never sufficient.

In [ ]:
from cybercommons import ir
import time

t0 = time.time()
tl = ir.Timeline()
tl.add(t0,      "alice", "alice",       "login",      "console")
tl.add(t0 + 30, "alice", "patch-agent", "read_file",  "/work/.env")
tl.add(t0 + 31, "alice", "patch-agent", "http_get",   "https://collect.example.com/")
tl.add(t0 + 95, "alice", "alice",       "logout",     "console")
print(tl.render())

A reconstruction from this alone says: alice logged in, read a secrets file, posted it externally, and logged out. Every sentence is supported by the logs and the conclusion is wrong.

In [ ]:
r = ir.reconstruct(tl)
for k, v in r.items():
    print(f"{k:22s} {v}")

### Expect

The timeline reads as a single human actor. `reconstruct` reports BROKEN attribution, two misattributed lines, `patch-agent` as a hidden actor, and the consequence for containment.

### Your turn

What single field, added to these log lines, would have made the reconstruction correct? Now check whether your logs have it.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/D2.1.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*